# Chapter 6 · Shor's Algorithm

## Objectives

1. Understand the reduction of the factoring problem to finding the order of $a$ modulo $N$.
2. Implement the phase estimation circuit for factoring $N = 15$.
3. Perform the classical post-processing (continued fractions) to extract the order.
4. Verify that the correct factors are obtained.

---

## 6.1 Algorithm structure

Shor's algorithm factors $N$ in time $O((\log N)^3)$ by combining:

**Classical pre-step:** Choose random $a$ with $1 < a < N$ and $\gcd(a, N) = 1$.  
**Quantum step:** Find the order $r$ of $a$ modulo $N$, i.e., the smallest $r > 0$ such that $a^r \equiv 1 \pmod{N}$.  
**Classical post-step:** If $r$ is even and $a^{r/2} \not\equiv -1 \pmod{N}$, then

$$p = \gcd(a^{r/2} - 1, N) \quad \text{and} \quad q = \gcd(a^{r/2} + 1, N)$$

are non-trivial factors of $N$.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..'))

import numpy as np
import matplotlib.pyplot as plt
from math import gcd
from fractions import Fraction
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.quantum_info import Operator, Statevector
from qiskit_aer import AerSimulator
from src.visualization import QuantumVisualization

print('Dependencies loaded.')

## 6.2 Classical post-processing

In [ ]:
def classical_postprocessing(measured_phase: int, n_count: int,
                              N: int, a: int) -> dict:
    """Post-processing of Shor's algorithm.

    Applies the continued fractions algorithm to the measured phase to
    obtain the order r, then computes the factors of N.

    Parameters
    ----------
    measured_phase : int
        Integer measured in the counting qubits.
    n_count : int
        Number of counting qubits (precision = 2^n_count).
    N : int
        Number to factor.
    a : int
        Chosen base.

    Returns
    -------
    dict with keys: phase_ratio, r, factors, success.
    """
    # Estimated phase
    phase_ratio = measured_phase / (2 ** n_count)

    # Continued fraction approximation
    frac = Fraction(phase_ratio).limit_denominator(N)
    r = frac.denominator

    result = {
        'phase_ratio': phase_ratio,
        'r': r,
        'factors': [],
        'success': False,
    }

    if r == 0 or r % 2 != 0:
        return result

    # Factor candidates
    ar2 = pow(a, r // 2, N)
    if ar2 == N - 1:   # a^{r/2} ≡ -1 (mod N)
        return result

    p = gcd(ar2 - 1, N)
    q = gcd(ar2 + 1, N)

    factors = [f for f in [p, q] if 1 < f < N]
    result['factors'] = factors
    result['success'] = len(factors) > 0

    return result


# Manual example: a=2, N=15, r=4 (known)
res = classical_postprocessing(measured_phase=4, n_count=3, N=15, a=2)
print('Post-processing (manual example):')
for k, v in res.items():
    print(f'  {k}: {v}')

## 6.3 Quantum phase estimation circuit for N=15, a=2

For $N=15$ and $a=2$, the order is $r=4$ because $2^4 = 16 \equiv 1 \pmod{15}$. We implement the modular multiplication oracle $|x\rangle \mapsto |ax \bmod N\rangle$ for this specific case.

In [ ]:
def c_amod15(a: int, power: int) -> QuantumCircuit:
    """Controlled gate U^{2^power} for a mod 15.

    Only implements cases a ∈ {2, 4, 7, 8, 11, 13}.
    """
    if a not in [2, 4, 7, 8, 11, 13]:
        raise ValueError(f'a={a} not implemented.')

    U = QuantumCircuit(4, name=f'U_{a}^{2**power}')
    # Apply the corresponding permutation 'power' times
    for _ in range(power):
        if a == 2:
            U.swap(0, 1); U.swap(1, 2); U.swap(2, 3)
        elif a == 7:
            U.swap(2, 3); U.swap(1, 2); U.swap(0, 1)
        elif a == 8:
            U.swap(2, 3); U.swap(1, 2); U.swap(0, 1)
        elif a == 4:
            U.swap(1, 3); U.swap(0, 2)
        elif a == 11:
            U.swap(0, 2); U.swap(1, 3)
            U.x(range(4))
        elif a == 13:
            U.swap(0, 3); U.swap(1, 2)
            U.x(range(4))

    U_gate = U.to_gate().control(1)
    qc = QuantumCircuit(5)
    qc.append(U_gate, range(5))
    return qc


def shor_circuit_n15(a: int = 2, n_count: int = 8) -> QuantumCircuit:
    """Shor's circuit for N=15.

    Parameters
    ----------
    a : int
        Base (must be coprime with 15).
    n_count : int
        Number of counting qubits (more qubits = higher precision).
    """
    from notebooks.ch04_fourier_cuantica import qft  # import locally
    # import the QFT function defined here
    pass


# ── Direct implementation with Qiskit ──────────────────────────────
def build_shor_n15(a: int = 2, n_count: int = 8) -> QuantumCircuit:
    """Shor's circuit for N=15 (direct implementation)."""
    from qiskit.circuit.library import QFT

    # Counting register (n_count qubits) + working register (4 qubits)
    q_count = QuantumRegister(n_count, 'count')
    q_aux   = QuantumRegister(4, 'aux')
    cr      = ClassicalRegister(n_count, 'meas')
    qc = QuantumCircuit(q_count, q_aux, cr)

    # Initial state of the auxiliary register = |1〉
    qc.x(q_aux[0])

    # Hadamard on counting qubits
    qc.h(q_count)
    qc.barrier()

    # Controlled U^{2^j} gates
    for j in range(n_count):
        controlled_U = c_amod15(a, 2**j)
        # connect counting qubit j with the auxiliary register
        qc.append(controlled_U, [q_count[j]] + list(q_aux))
    qc.barrier()

    # Inverse QFT on the counting register
    iqft = QFT(n_count, inverse=True, do_swaps=True)
    qc.append(iqft, q_count)

    # Measurement
    qc.measure(q_count, cr)
    return qc


n_count = 8
a = 2
qc_shor = build_shor_n15(a=a, n_count=n_count)
print(f'Shor circuit for N=15, a={a}, n_count={n_count}:')
print(f'  Depth: {qc_shor.depth()}')
print(f'  Number of operations: {qc_shor.size()}')

In [ ]:
# Run the circuit
backend = AerSimulator()
job = backend.run(qc_shor, shots=2048)
counts = job.result().get_counts()

print(f'Results (top 10):')
sorted_counts = sorted(counts.items(), key=lambda x: -x[1])[:10]
for state, cnt in sorted_counts:
    phase_int = int(state, 2)
    result = classical_postprocessing(phase_int, n_count, N=15, a=a)
    factors_str = f' → factors: {result["factors"]}' if result['success'] else ''
    print(f'  {state} (={phase_int:3d}): {cnt:4d} occurrences{factors_str}')

fig = QuantumVisualization.plot_histogram(
    counts,
    title=f'Shor N=15, a={a}: distribution of measured phases',
    color='#d2a8ff',
)
plt.show()

## 6.4 Proposed exercises

1. Repeat the experiment with $a = 7$ and $a = 13$ for $N = 15$. Do you obtain the same factors?

2. Explain why the classical continued fractions algorithm can fail and how many quantum repetitions are needed on average to guarantee success.

3. How many physical qubits would be needed to factor RSA-2048 with Shor's algorithm, assuming quantum error correction?

4. Implement the full classical reduction step: given an arbitrary $N$, choose a random $a$ and verify whether $\gcd(a, N) \neq 1$ before calling the quantum circuit.